# Hyperliquid vault-of-vaults — initial v11 backtests

Replay `hyper-ai-v11.py` through the production strategy engine over **two periods**: the short source-reference window (1 January–5 October 2026, end exclusive) and all available history for the selected Hypercore vault universe. Both start independently with $150,000 and use the source's two-day decision cycle, six-vault limit, indicators, fee model and guard record.

The long window starts at the first valid vault observation, excluding the older benchmark-pair history. Cash-only days before the strategy can select vaults remain part of the long-period metrics. The engine aligns decisions to its fixed two-day schedule. For the long run it rounds the requested 1 March 2023 start down to a cash-only 28 February decision; the first trade is on 31 May 2023. The windows overlap, but fresh initial state means their holdings can differ. The report includes a side-by-side standard performance table, equity curves, quarterly profit heatmaps and absolute allocation in USD, including reserve cash.

The universe is selected by today's v11 rules and uses today's guard snapshot throughout. Historical deposit-state snapshots only begin on 11 April 2026; earlier dates assume deposits are open. This is a historical replay of the current strategy and dataset, with those limitations.

The source path and SHA-256 below identify the exact strategy. Only the short period has a published reference result against which replication is checked.

Execution fixes needed for the long history: the framework now handles all-zero sizing weights during indicator warm-up, and v11's cost-basis quantity assertion allows floating-point rounding at large share counts. The latter changes only the assertion tolerance, not cost-basis or fee calculations. Fee-adjusted vault execution diagnostics and local kernel IPC remove the original warnings at source; no warning filters are applied.

## Restored-data results — 7 October 2026

The source replay now gives 41.20% CAGR, 1.96 Sharpe and −6.40% maximum drawdown. Final equity is $194,520.81 versus the earlier $194,928.64 reference. The restored policy inputs remove most of the previous gap. The first 558 chronologically ordered trades match exactly; remaining differences begin in September.

### Strategy comparison — 2026-01-01 → 2026-10-05 (end exclusive, UTC)

Fresh engine backtests on the same pinned publication and merged readers, each starting with $150,000. Full-history runs are retained separately in the saved outputs.

| Strategy | Final equity (USD) | CAGR | Sharpe | Maximum drawdown | Trades |
| --- | --- | --- | --- | --- | --- |
| Source v11 | \$194,520.81 | 41.20% | 1.96 | -6.40% | 581 |

Both periods completed. Historical benchmark differences are separate from mandatory equality between controls on identical inputs. See [the rerun report](restored-data-rerun-2026-10-07.md) for input and clock provenance.


### Restored leader history rerun — 7 October 2026

This execution uses the freshly downloaded, pinned repaired dataset and the merged leader-share and recorded-cap readers. Strategy settings, searches and period boundaries are unchanged. Historical benchmark differences are displayed separately from mandatory source/control equality on the same inputs. Generated results are isolated under `_artifacts-restored-2026-10-07/01`.


In [1]:
import datetime
from copy import deepcopy
import hashlib
import importlib.util
import logging
import os
from pathlib import Path

import pandas as pd
import plotly.express as px
from IPython.display import display
from dotenv import load_dotenv

from tradingstrategy.client import Client
from tradeexecutor.backtest.backtest_runner import run_backtest_inline
from tradeexecutor.strategy.execution_context import notebook_execution_context
from tradeexecutor.strategy.parameters import StrategyParameters
from tradeexecutor.strategy.reserve_currency import ReserveCurrency
from tradeexecutor.statistics.key_metric import (
    calculate_cagr,
    calculate_max_drawdown,
    calculate_sharpe,
    calculate_sortino,
)
from tradeexecutor.visual.equity_curve import calculate_equity_curve, calculate_returns
from tradeexecutor.utils.notebook import setup_charting_and_output, OutputMode

load_dotenv(override=True)
setup_charting_and_output(OutputMode.static, image_format="png", width=1500, height=1000)

STRATEGY_PATH = Path("/Users/moo/code/strategies/strategy/hyper-ai-v11.py")
GUARD_PATH = Path("/Users/moo/code/strategies/deploy/hyper-ai-v10-new-guard-vault-info.json")
os.environ["HYPER_AI_GUARD_CONFIG_FILE"] = str(GUARD_PATH)
assert STRATEGY_PATH.is_file(), STRATEGY_PATH
assert GUARD_PATH.is_file(), GUARD_PATH

spec = importlib.util.spec_from_file_location("hyper_ai_v11", STRATEGY_PATH)
strategy = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy)
source_sha256 = hashlib.sha256(STRATEGY_PATH.read_bytes()).hexdigest()
parameters = StrategyParameters.from_class(strategy.Parameters)
client = Client.create_jupyter_client()

display(pd.DataFrame([
    {"Input": "Strategy source", "Value": str(STRATEGY_PATH)},
    {"Input": "Strategy SHA-256", "Value": source_sha256},
    {"Input": "Guard record", "Value": str(GUARD_PATH)},
    {"Input": "Decision cycle", "Value": str(parameters.cycle_duration)},
    {"Input": "Initial equity (USD)", "Value": parameters.initial_cash},
]))
from score_matrix_visualisation import with_backtest_period

def backtest_period_label(start, end):
    return f"{start:%Y-%m-%d} → {end:%Y-%m-%d} (end exclusive, UTC)"


,Input,Value
0,Strategy source,/Users/moo/code/strategies/strategy/hyper-ai-v...
1,Strategy SHA-256,5ed8a4c00f1edb14ae9af40ac3d94d3622587445921d1e...
2,Guard record,/Users/moo/code/strategies/deploy/hyper-ai-v10...
3,Decision cycle,CycleDuration.cycle_2d
4,Initial equity (USD),150000


## Prepare the universe and indicators

Load v11's full indicator history once, then derive the long window from valid observations for the selected vaults. Both runs use identical indicators and an exclusive end boundary of 5 October 2026, retaining only completed daily candles.

In [2]:
from tradeexecutor.strategy.pandas_trader.trading_universe_input import CreateTradingUniverseInput
from tradeexecutor.strategy.universe_model import UniverseOptions
from tradeexecutor.strategy.pandas_trader.indicator import calculate_and_load_indicators_inline

universe_options = UniverseOptions.from_strategy_parameters_class(
    strategy.Parameters, notebook_execution_context
)
universe_input = CreateTradingUniverseInput(
    execution_context=notebook_execution_context,
    client=client,
    timestamp=None,
    parameters=parameters,
    universe_options=universe_options,
    execution_model=None,
)
print("Loading v11 universe and price history; source run used the Trading Strategy vault dataset.")
import sys
sys.path.insert(0, "/Users/moo/code/getting-started/scratchpad/hyperliquid-vov-2nd")
from repaired_data_replay import load_repaired_universe, INDICATOR_DIR, repaired_indicator_storage
strategy_universe = load_repaired_universe(strategy, universe_input)
print(f"Loaded {strategy_universe.get_pair_count()} pairs; calculating indicators with one worker.")
indicator_data = calculate_and_load_indicators_inline(
    indicator_storage_path=INDICATOR_DIR,
    strategy_universe=strategy_universe,
    parameters=parameters,
    create_indicators=strategy.create_indicators,
    execution_context=notebook_execution_context,
    max_workers=1,
)
print(f"Indicator combinations: {len(indicator_data.indicator_combinations)}")

# Exclude benchmark pairs when determining the available vault history.
vault_history = []
for pair in strategy_universe.iterate_pairs():
    if pair.is_vault():
        candles = strategy_universe.data_universe.candles.get_candles_by_pair(pair.internal_id)
        valid = candles.loc[candles["close"].gt(0)]
        if len(valid):
            vault_history.append({
                "Pair id": pair.internal_id,
                "First observation": valid.index.min(),
                "Last observation": valid.index.max(),
            })
vault_history_df = pd.DataFrame(vault_history)
long_start = pd.Timestamp(vault_history_df["First observation"].min()).to_pydatetime()
assert long_start < parameters.backtest_start
# Preserve the source's exclusive end boundary: no incomplete 5 October candle.
periods = {
    "Short (source reference)": (parameters.backtest_start, parameters.backtest_end),
    "Long (all available vault history)": (long_start, parameters.backtest_end),
}
display(pd.DataFrame([
    {"Period": label, "Start (UTC)": start, "End (exclusive, UTC)": end,
     "Calendar days": (end - start).days, "Vaults with history": len(vault_history_df)}
    for label, (start, end) in periods.items()
]))

,Period,Start (UTC),"End (exclusive, UTC)",Calendar days,Vaults with history
0,Short (source reference),2026-01-01,2026-10-05,277,514
1,Long (all available vault history),2023-03-01,2026-10-05,1314,514


## Backtests and key metrics

Run each period with fresh portfolio state and initial cash. Reset mutable vault fee estimates between runs. Returns, CAGR, Sharpe, Sortino and drawdown use the daily-resampled convention recorded in the source reference; initial idle days are included.

In [3]:
runs = {}
for label, (start, end) in periods.items():
    run_parameters = deepcopy(parameters)
    run_parameters["backtest_start"] = start
    run_parameters["backtest_end"] = end
    # Reset fees mutated by the preceding run; each engine run starts with fresh cash and state.
    strategy.apply_vault_redemption_capital_fee(
        strategy_universe, float(run_parameters.vault_redemption_capital_fee)
    )
    print(f"Starting {label}: {start.date()} to {end.date()} (exclusive); allow a few minutes.")
    result = run_backtest_inline(
        engine_version=strategy.trading_strategy_engine_version,
        client=client,
        decide_trades=strategy.decide_trades,
        universe=strategy_universe,
        indicator_storage=repaired_indicator_storage(strategy_universe),
        indicator_combinations=indicator_data.indicator_combinations,
        parameters=run_parameters,
        cycle_duration=run_parameters.cycle_duration,
        initial_deposit=run_parameters.initial_cash,
        reserve_currency=ReserveCurrency.usdc,
        trade_routing=run_parameters.routing,
        max_slippage=run_parameters.slippage_tolerance_pct,
        start_at=start,
        end_at=end,
        max_workers=1,
        name=f"Hyper AI v11 — {label}",
        strategy_logging=False,
    )
    state = result.state
    equity = calculate_equity_curve(state)
    # Match the daily-resampled metric convention documented in hyper-ai-v11.py.
    daily_returns = calculate_returns(equity).resample("1D").sum(min_count=1).fillna(0.0).dropna()
    positions = [p for p in state.portfolio.get_all_positions() if not p.is_credit_supply()]
    cycle_stamps = sorted(s.calculated_at for s in state.stats.portfolio)
    trades = list(state.portfolio.get_all_trades())
    position_counts_by_cycle = {}
    for position_stats in state.stats.positions.values():
        for snapshot in position_stats:
            if snapshot.value > 0:
                position_counts_by_cycle[snapshot.calculated_at] = position_counts_by_cycle.get(snapshot.calculated_at, 0) + 1
    metrics = {
        "Initial equity (USD)": float(run_parameters.initial_cash),
        "Final equity (USD)": float(equity.iloc[-1]),
        "Cumulative return": float((1.0 + daily_returns).prod() - 1.0),
        "CAGR": float(calculate_cagr(daily_returns)),
        "Sharpe": float(calculate_sharpe(daily_returns, periods=365)),
        "Sortino": float(calculate_sortino(daily_returns, periods=365)),
        "Maximum drawdown": float(calculate_max_drawdown(daily_returns)),
        "Decision cycles": len(cycle_stamps),
        "Distinct positions": len(positions),
        "Maximum vaults held": max(position_counts_by_cycle.values(), default=0),
        "Trades": len(trades),
    }
    runs[label] = {"result": result, "parameters": run_parameters, "equity": equity,
                   "daily_returns": daily_returns, "metrics": metrics,
                   "First decision": cycle_stamps[0], "Last decision": cycle_stamps[-1],
                   "First trade": min((t.opened_at for t in trades), default=None)}

metrics_df = pd.DataFrame({label: run["metrics"] for label, run in runs.items()})
operational_rows = ["Initial equity (USD)", "Final equity (USD)", "Decision cycles",
                    "Distinct positions", "Maximum vaults held", "Trades"]
display(metrics_df.loc[operational_rows].style.format("{:,.2f}"))
display(pd.DataFrame([
    {"Period": label, "First decision": run["First decision"],
     "Last decision": run["Last decision"], "First trade": run["First trade"]}
    for label, run in runs.items()
]))
from repaired_data_replay import RUN_ROOT, save_metrics
save_metrics(runs,RUN_ROOT/"01")


,Short (source reference),Long (all available vault history)
Initial equity (USD),"150,000.00","150,000.00"
Final equity (USD),"194,520.81","254,103.35"
Decision cycles,139.00,658.00
Distinct positions,101.00,214.00
Maximum vaults held,6.00,6.00
Trades,581.00,"1,074.00"


,Period,First decision,Last decision,First trade
0,Short (source reference),2026-01-01,2026-10-04,2026-01-01
1,Long (all available vault history),2023-02-28,2026-10-04,2023-05-31


## Key metrics comparison

Use trade-executor's standard QuantStats performance table, as in the earlier backtest notebooks. The columns compare **v11 over the short reference period** with **v11 over the full history**. Calculate each column on its own daily returns before combining the tables, so the short run is not padded with cash-only returns from the long window. This compares two periods of the same strategy.

The table includes return, CAGR, volatility, Sharpe, Sortino, Calmar, drawdown, recovery and other risk metrics. The source-reference checks below retain the original rounding tolerances.

In [4]:
from tradeexecutor.analysis.advanced_metrics import calculate_advanced_metrics, AdvancedMetricsMode

performance_tables = []
for label, run in runs.items():
    table = calculate_advanced_metrics(
        run["daily_returns"],
        mode=AdvancedMetricsMode.full,
        periods_per_year=365,
        convert_to_daily=False,
        display=True,
    )
    performance_tables.append(table.rename(columns={"Strategy": f"v11 — {label}"}))
key_metrics_comparison = pd.concat(performance_tables, axis=1)
key_metrics_comparison.index.name = "Metric"
display(key_metrics_comparison)

,v11 — Short (source reference),v11 — Long (all available vault history)
Metric,,
Start Period,2026-01-01,2023-02-28
End Period,2026-10-04,2026-10-04
Risk-Free Rate,0.0%,0.0%
Time in Market,50.0%,31.0%
Cumulative Return,29.81%,69.4%
CAGR﹪,41.2%,15.77%
Sharpe,1.96,1.24
Prob. Sharpe Ratio,96.5%,98.89%
Smart Sharpe,1.94,1.23


## Equity curves

In [5]:
for label, run in runs.items():
    equity_frame = run["equity"].rename("Portfolio equity (USD)").rename_axis("Date").reset_index()
    fig = px.line(equity_frame, x="Date", y="Portfolio equity (USD)", title=f"Hyper AI v11 — {label}")
    fig.update_layout(width=1500, height=700)
    with_backtest_period(fig, backtest_period_label(run["parameters"].backtest_start, run["parameters"].backtest_end))
    fig.show(width=1500, height=700)

## Quarterly profit heatmaps

Use the existing trade-executor returns heatmap with calendar-quarter aggregation. Each cell shows compounded portfolio profit/loss in percent, including changes in the value of open positions. The calculation uses the same daily returns as the metrics table.

Boundary quarters contain only observed dates: long-period 2023 Q1 starts with the cash-only 28 February cycle, and 2026 Q4 covers 1–4 October in both runs. These are partial quarters. Blank cells are outside the run's window; 0% means the observed portfolio return was zero.

In [6]:
from matplotlib import pyplot as plt
from tradeexecutor.visual.equity_curve import visualise_returns_over_time

for label, run in runs.items():
    fig = visualise_returns_over_time(run["daily_returns"], period="quarter")
    fig.axes[0].set_title(f"Quarterly portfolio profit/loss — v11 — {label}" + "\nBacktest: " + backtest_period_label(run["parameters"].backtest_start, run["parameters"].backtest_end))
    display(fig)
    plt.close(fig)

## Absolute asset allocation

Each stacked chart shows the USD value of every vault holding and reserve cash. Values are absolute, rather than normalised to 100%.

In [7]:
from tradeexecutor.strategy.chart.renderer import ChartBacktestRenderingSetup
from tradeexecutor.strategy.chart.standard.weight import equity_curve_by_asset

for label, run in runs.items():
    result = run["result"]
    run_parameters = run["parameters"]
    charts = strategy.create_charts(
        timestamp=None,
        parameters=run_parameters,
        strategy_universe=result.strategy_universe,
        execution_context=notebook_execution_context,
    )
    chart_renderer = ChartBacktestRenderingSetup(
        registry=charts,
        strategy_input_indicators=result.indicators,
        state=result.state,
        backtest_start_at=run_parameters.backtest_start,
        backtest_end_at=run_parameters.backtest_end,
    )
    asset_weights_fig = chart_renderer.render(equity_curve_by_asset)
    asset_weights_fig.update_layout(title=f"Absolute asset allocation (USD) — {label}", width=1500, height=900)
    # Keep every vault in the static legend visible across the longer history.
    chart_height = max(900, 170 + 20 * len(asset_weights_fig.data))
    with_backtest_period(asset_weights_fig, backtest_period_label(run_parameters.backtest_start, run_parameters.backtest_end))
    asset_weights_fig.show(width=1500, height=chart_height)

## Short-period replication check

Compare only the short run with the rounded reference metrics in `hyper-ai-v11.py`. Tolerances allow rounding; the final assertion fails on meaningful differences. The long run is a separate experiment and has no published v11 reference.

In [8]:
metrics = runs["Short (source reference)"]["metrics"]
reference = {
    "Final equity (USD)": 194_929.0,
    "Cumulative return": 0.3008,
    "CAGR": 0.4159,
    "Sharpe": 2.00,
    "Sortino": 3.56,
    "Maximum drawdown": -0.0640,
    "Decision cycles": 139,
    "Distinct positions": 101,
    "Maximum vaults held": 6,
}
tolerances = {
    "Final equity (USD)": 2.0,
    "Cumulative return": 0.00015,
    "CAGR": 0.00015,
    "Sharpe": 0.015,
    "Sortino": 0.015,
    "Maximum drawdown": 0.00015,
    "Decision cycles": 0,
    "Distinct positions": 0,
    "Maximum vaults held": 0,
}
comparison_rows = []
for metric, expected in reference.items():
    actual = metrics[metric]
    tolerance = tolerances[metric]
    comparison_rows.append({
        "Metric": metric,
        "Reference": expected,
        "Rerun": actual,
        "Difference": actual - expected,
        "Tolerance": tolerance,
        "Match": abs(actual - expected) <= tolerance,
    })
replication_df = pd.DataFrame(comparison_rows)
display(replication_df)
replication_matches = bool(replication_df["Match"].all())
print(f"Reference reproduction: {'PASS' if replication_matches else 'DIFFERENCES FOUND'}")
# Historical benchmark differences are reported; same-input replication is checked separately.

,Metric,Reference,Rerun,Difference,Tolerance,Match
0,Final equity (USD),194929.0000,194520.812686,-408.187314,2.00000,False
1,Cumulative return,0.3008,0.298078,-0.002722,0.00015,False
2,CAGR,0.4159,0.412004,-0.003896,0.00015,False
3,Sharpe,2.0000,1.959590,-0.040410,0.01500,False
4,Sortino,3.5600,3.429825,-0.130175,0.01500,False
5,Maximum drawdown,-0.0640,-0.064031,-0.000031,0.00015,True
6,Decision cycles,139.0000,139.000000,0.000000,0.00000,True
7,Distinct positions,101.0000,101.000000,0.000000,0.00000,True
8,Maximum vaults held,6.0000,6.000000,0.000000,0.00000,True
